**Generative AI Use**: For the purposes of the assignments, the use of generative AI is subject to the same policies regarding collaboration. Just as with other collaborators, each student must write down the solutions independently of the output of the interaction and the submission should include a note denoting the nature of the collaboration. The use of generative AI tools to substantially complete sections of the assignments is not in line with the spirit of the assignments, and would be a violation of the [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code).
**关于生成式 AI 的使用**：就本作业而言，使用生成式 AI 与其他形式的协作遵循同样的政策。正如对待其他协作者一样，每位学生都必须独立于交互产生的输出来写下自己的解答，并且提交材料中应注明该协作的性质。使用生成式 AI 工具来实质性地完成作业的各个部分，不符合本作业的精神，将构成对[荣誉准则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)的违反。


In [ ]:
# This mounts your Google Drive to the Colab VM.
# 这一步把你的 Google Drive 挂载到 Colab 虚拟机上。
from google.colab import drive
drive.mount('/content/drive')

# TODO: Enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'cs231n/assignments/assignment1/'
# TODO: 在这里填入你 Drive 中存放已解压作业文件夹的那个目录名，
# 例如 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
# Drive 挂载好之后，这一步保证 Colab 虚拟机里的 Python 解释器
# 能够从该目录中加载 python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# This downloads the CIFAR-10 dataset to your Drive
# if it doesn't already exist.
# 如果 CIFAR-10 数据集还不存在，这一步会把它下载到你的 Drive 里。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# Fully-Connected Neural Nets
# 全连接神经网络
In this exercise we will implement fully-connected networks using a modular approach. For each layer we will implement a `forward` and a `backward` function. The `forward` function will receive inputs, weights, and other parameters and will return both an output and a `cache` object storing data needed for the backward pass, like this:
在本练习中，我们用模块化的方式实现全连接网络。对每一层都实现一个 `forward` 函数和一个 `backward` 函数。`forward` 函数接收输入、权重和其他参数，返回输出以及一个保存反向传播所需数据的 `cache` 对象，就像这样：

```python
def layer_forward(x, w):
  """ Receive inputs x and weights w """
  # 接收输入 x 和权重 w
  # Do some computations ...
  # 做一些计算……
  z = # ... some intermediate value
  # ……某个中间值
  # Do some more computations ...
  # 再做些计算……
  out = # the output
  # 输出
   
  cache = (x, w, z, out) # Values we need to compute gradients
                        # 计算梯度所需的那些值
   
  return out, cache
```

The backward pass will receive upstream derivatives and the `cache` object, and will return gradients with respect to the inputs and weights, like this:
反向传播接收上游传回的导数以及 `cache` 对象，返回损失对输入和权重的梯度，就像这样：

```python
def layer_backward(dout, cache):
  """
  Receive dout (derivative of loss with respect to outputs) and cache,
  and compute derivative with respect to inputs.
  """
  # 接收 dout（损失对输出的导数）和 cache，
  # 计算损失对输入的导数。
  # Unpack cache values
  # 解包 cache 中的值
  x, w, z, out = cache
  
  # Use values in cache to compute derivatives
  # 用 cache 中的值计算导数
  dx = # Derivative of loss with respect to x
  # 损失对 x 的导数
  dw = # Derivative of loss with respect to w
  # 损失对 w 的导数
  
  return dx, dw
```

After implementing a bunch of layers this way, we will be able to easily combine them to build classifiers with different architectures.
这样实现完一堆层之后，我们就能轻松地把它们组合起来，搭建不同架构的分类器。
  

In [ ]:
# As usual, a bit of setup
# 照例，做一点初始化设置
from __future__ import print_function
import time
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.fc_net import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
                                             # 设置图形的默认尺寸
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading external modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
# 用于自动重新加载外部模块
# 参见 http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

def rel_error(x, y):
  # 返回相对误差
  """ returns relative error """
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# Load the (preprocessed) CIFAR10 data.
# 加载（已预处理过的）CIFAR10 数据。

data = get_CIFAR10_data()
for k, v in list(data.items()):
  print(('%s: ' % k, v.shape))

# Affine layer: forward
# 仿射层：前向传播
Open the file `cs231n/layers.py` and implement the `affine_forward` function.
打开文件 `cs231n/layers.py`，实现 `affine_forward` 函数。

Once you are done you can test your implementaion by running the following:
实现完成后，可以运行下面的代码来测试你的实现：

In [ ]:
# Test the affine_forward function
# 测试 affine_forward 函数

num_inputs = 2
input_shape = (4, 5, 6)
output_dim = 3

input_size = num_inputs * np.prod(input_shape)
weight_size = output_dim * np.prod(input_shape)

x = np.linspace(-0.1, 0.5, num=input_size).reshape(num_inputs, *input_shape)
w = np.linspace(-0.2, 0.3, num=weight_size).reshape(np.prod(input_shape), output_dim)
b = np.linspace(-0.3, 0.1, num=output_dim)

out, _ = affine_forward(x, w, b)
correct_out = np.array([[ 1.49834967,  1.70660132,  1.91485297],
                        [ 3.25553199,  3.5141327,   3.77273342]])

# Compare your output with ours. The error should be around e-9 or less.
# 把你的输出与我们的结果比较。误差应该在 e-9 左右或更小。
print('Testing affine_forward function:')
print('difference: ', rel_error(out, correct_out))

# Affine layer: backward
# 仿射层：反向传播
Now implement the `affine_backward` function and test your implementation using numeric gradient checking.
现在实现 `affine_backward` 函数，并用数值梯度检查来测试你的实现。

In [ ]:
# Test the affine_backward function
# 测试 affine_backward 函数
np.random.seed(231)
x = np.random.randn(10, 2, 3)
w = np.random.randn(6, 5)
b = np.random.randn(5)
dout = np.random.randn(10, 5)

dx_num = eval_numerical_gradient_array(lambda x: affine_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_forward(x, w, b)[0], b, dout)

_, cache = affine_forward(x, w, b)
dx, dw, db = affine_backward(dout, cache)

# The error should be around e-10 or less
# 误差应该在 e-10 左右或更小
print('Testing affine_backward function:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

# ReLU activation: forward
# ReLU 激活函数：前向传播
Implement the forward pass for the ReLU activation function in the `relu_forward` function and test your implementation using the following:
在 `relu_forward` 函数中实现 ReLU 激活函数的前向传播，并用下面的代码测试你的实现：

In [ ]:
# Test the relu_forward function
# 测试 relu_forward 函数

x = np.linspace(-0.5, 0.5, num=12).reshape(3, 4)

out, _ = relu_forward(x)
correct_out = np.array([[ 0.,          0.,          0.,          0.,        ],
                        [ 0.,          0.,          0.04545455,  0.13636364,],
                        [ 0.22727273,  0.31818182,  0.40909091,  0.5,       ]])

# Compare your output with ours. The error should be on the order of e-8
# 把你的输出与我们的结果比较。误差应该在 e-8 量级
print('Testing relu_forward function:')
print('difference: ', rel_error(out, correct_out))

# ReLU activation: backward
# ReLU 激活函数：反向传播
Now implement the backward pass for the ReLU activation function in the `relu_backward` function and test your implementation using numeric gradient checking:
现在在 `relu_backward` 函数中实现 ReLU 激活函数的反向传播，并用数值梯度检查来测试你的实现：

In [ ]:
np.random.seed(231)
x = np.random.randn(10, 10)
dout = np.random.randn(*x.shape)

dx_num = eval_numerical_gradient_array(lambda x: relu_forward(x)[0], x, dout)

_, cache = relu_forward(x)
dx = relu_backward(dout, cache)

# The error should be on the order of e-12
# 误差应该在 e-12 量级
print('Testing relu_backward function:')
print('dx error: ', rel_error(dx_num, dx))

## Inline Question 1:
## 课堂提问 1：

We've only asked you to implement ReLU, but there are a number of different activation functions that one could use in neural networks, each with its pros and cons. In particular, an issue commonly seen with activation functions is getting zero (or close to zero) gradient flow during backpropagation. Which of the following activation functions have this problem? If you consider these functions in the one dimensional case, what types of input would lead to this behaviour?
我们只要求你实现 ReLU，但神经网络中可以使用的激活函数有很多，各有优缺点。激活函数的一个常见问题是：反向传播时梯度流变成零（或接近零）。下面哪些激活函数有这个问题？如果把这些函数看作一维情形，什么样的输入会导致这种行为？
1. Sigmoid
2. ReLU
3. Leaky ReLU

$\color{blue}{\textit Your Answer:}$ *Fill this in*
**你的答案：** *在这里填写。*




# "Sandwich" layers
# “三明治”层
There are some common patterns of layers that are frequently used in neural nets. For example, affine layers are frequently followed by a ReLU nonlinearity. To make these common patterns easy, we define several convenience layers in the file `cs231n/layer_utils.py`.
神经网络中经常使用一些常见的层组合模式。例如，仿射层后面常常紧跟一个 ReLU 非线性。为了让这些常见模式用起来方便，我们在文件 `cs231n/layer_utils.py` 中定义了几个便捷层。

For now take a look at the `affine_relu_forward` and `affine_relu_backward` functions, and run the following to numerically gradient check the backward pass:
现在先看一下 `affine_relu_forward` 和 `affine_relu_backward` 函数，然后运行下面的代码，对反向传播做数值梯度检查：

In [ ]:
from cs231n.layer_utils import affine_relu_forward, affine_relu_backward
np.random.seed(231)
x = np.random.randn(2, 3, 4)
w = np.random.randn(12, 10)
b = np.random.randn(10)
dout = np.random.randn(2, 10)

out, cache = affine_relu_forward(x, w, b)
dx, dw, db = affine_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: affine_relu_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_relu_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_relu_forward(x, w, b)[0], b, dout)

# Relative error should be around e-10 or less
# 相对误差应该在 e-10 左右或更小
print('Testing affine_relu_forward and affine_relu_backward:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

# Loss layers: Softmax
# 损失层：Softmax
Now implement the loss and gradient for softmax in the `softmax_loss` function in `cs231n/layers.py`. These should be similar to what you implemented in `cs231n/classifiers/softmax.py`. Other loss functions (e.g. `svm_loss`) can also be implemented in a modular way, however, it is not required for this assignment.
现在在 `cs231n/layers.py` 的 `softmax_loss` 函数中实现 softmax 的损失和梯度。这些内容应该与你在 `cs231n/classifiers/softmax.py` 中实现过的类似。其他损失函数（例如 `svm_loss`）也可以用模块化的方式实现，但本作业不作要求。

You can make sure that the implementations are correct by running the following:
运行下面的代码可以确认实现是否正确：

In [ ]:
np.random.seed(231)
num_classes, num_inputs = 10, 50
x = 0.001 * np.random.randn(num_inputs, num_classes)
y = np.random.randint(num_classes, size=num_inputs)


dx_num = eval_numerical_gradient(lambda x: softmax_loss(x, y)[0], x, verbose=False)
loss, dx = softmax_loss(x, y)

# Test softmax_loss function. Loss should be close to 2.3 and dx error should be around e-8
# 测试 softmax_loss 函数。损失应该接近 2.3，dx 误差应该在 e-8 左右
print('\nTesting softmax_loss:')
print('loss: ', loss)
print('dx error: ', rel_error(dx_num, dx))

# Two-layer network
# 两层神经网络
Open the file `cs231n/classifiers/fc_net.py` and complete the implementation of the `TwoLayerNet` class. Read through it to make sure you understand the API. You can run the cell below to test your implementation.
打开文件 `cs231n/classifiers/fc_net.py`，完成 `TwoLayerNet` 类的实现。通读一遍，确保你理解了它的 API。可以运行下面的单元格来测试你的实现。

In [ ]:
np.random.seed(231)
N, D, H, C = 3, 5, 50, 7
X = np.random.randn(N, D)
y = np.random.randint(C, size=N)

std = 1e-3
model = TwoLayerNet(input_dim=D, hidden_dim=H, num_classes=C, weight_scale=std)

print('Testing initialization ... ')
W1_std = abs(model.params['W1'].std() - std)
b1 = model.params['b1']
W2_std = abs(model.params['W2'].std() - std)
b2 = model.params['b2']
assert W1_std < std / 10, 'First layer weights do not seem right'
assert np.all(b1 == 0), 'First layer biases do not seem right'
assert W2_std < std / 10, 'Second layer weights do not seem right'
assert np.all(b2 == 0), 'Second layer biases do not seem right'

print('Testing test-time forward pass ... ')
model.params['W1'] = np.linspace(-0.7, 0.3, num=D*H).reshape(D, H)
model.params['b1'] = np.linspace(-0.1, 0.9, num=H)
model.params['W2'] = np.linspace(-0.3, 0.4, num=H*C).reshape(H, C)
model.params['b2'] = np.linspace(-0.9, 0.1, num=C)
X = np.linspace(-5.5, 4.5, num=N*D).reshape(D, N).T
scores = model.loss(X)
correct_scores = np.asarray(
  [[11.53165108,  12.2917344,   13.05181771,  13.81190102,  14.57198434, 15.33206765,  16.09215096],
   [12.05769098,  12.74614105,  13.43459113,  14.1230412,   14.81149128, 15.49994135,  16.18839143],
   [12.58373087,  13.20054771,  13.81736455,  14.43418138,  15.05099822, 15.66781506,  16.2846319 ]])
scores_diff = np.abs(scores - correct_scores).sum()
assert scores_diff < 1e-6, 'Problem with test-time forward pass'

print('Testing training loss (no regularization)')
y = np.asarray([0, 5, 1])
loss, grads = model.loss(X, y)
correct_loss = 3.4702243556
assert abs(loss - correct_loss) < 1e-10, 'Problem with training-time loss'

model.reg = 1.0
loss, grads = model.loss(X, y)
correct_loss = 26.5948426952
assert abs(loss - correct_loss) < 1e-10, 'Problem with regularization loss'

# Errors should be around e-7 or less
# 误差应该在 e-7 左右或更小
for reg in [0.0, 0.7]:
  print('Running numeric gradient check with reg = ', reg)
  model.reg = reg
  loss, grads = model.loss(X, y)

  for name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    grad_num = eval_numerical_gradient(f, model.params[name], verbose=False)
    print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))

# Solver
# 求解器
Open the file `cs231n/solver.py` and read through it to familiarize yourself with the API. After doing so, use a `Solver` instance to train a `TwoLayerNet` that achieves about `36%` accuracy on the validation set.
打开文件 `cs231n/solver.py` 通读一遍，熟悉它的 API。然后使用一个 `Solver` 实例训练 `TwoLayerNet`，使其在验证集上达到约 `36%` 的准确率。

In [ ]:
input_size = 32 * 32 * 3
hidden_size = 50
num_classes = 10
model = TwoLayerNet(input_size, hidden_size, num_classes)
solver = None

##############################################################################
# TODO: Use a Solver instance to train a TwoLayerNet that achieves about 36% #
# accuracy on the validation set.                                            #
# TODO:                                                                      #
# 使用一个 Solver 实例训练 TwoLayerNet，使其在验证集上达到约 36% 的准确率。  #
##############################################################################

##############################################################################
#                             END OF YOUR CODE                               #
##############################################################################

# Debug the training
# 调试训练过程
With the default parameters we provided above, you should get a validation accuracy of about 0.36 on the validation set. This isn't very good.
用上面提供的默认参数，你应该会在验证集上得到约 0.36 的准确率。这个结果并不好。

One strategy for getting insight into what's wrong is to plot the loss function and the accuracies on the training and validation sets during optimization.
想要看清哪里出了问题，一个策略是在优化过程中画出损失函数以及训练集和验证集上的准确率。

Another strategy is to visualize the weights that were learned in the first layer of the network. In most neural networks trained on visual data, the first layer weights typically show some visible structure when visualized.
另一个策略是把网络第一层学到的权重可视化。在大多数用视觉数据训练的神经网络中，第一层的权重可视化后通常能看出一些可见的结构。

In [ ]:
# Run this cell to visualize training loss and train / val accuracy
# 运行这个单元格，把训练损失以及训练/验证准确率可视化

plt.subplot(2, 1, 1)
plt.title('Training loss')
plt.plot(solver.loss_history, 'o')
plt.xlabel('Iteration')

plt.subplot(2, 1, 2)
plt.title('Accuracy')
plt.plot(solver.train_acc_history, '-o', label='train')
plt.plot(solver.val_acc_history, '-o', label='val')
plt.plot([0.5] * len(solver.val_acc_history), 'k--')
plt.xlabel('Epoch')
plt.legend(loc='lower right')
plt.gcf().set_size_inches(15, 12)
plt.show()

In [ ]:
from cs231n.vis_utils import visualize_grid

# Visualize the weights of the network
# 可视化网络的权重

def show_net_weights(net):
    W1 = net.params['W1']
    W1 = W1.reshape(3, 32, 32, -1).transpose(3, 1, 2, 0)
    plt.imshow(visualize_grid(W1, padding=3).astype('uint8'))
    plt.gca().axis('off')
    plt.show()

show_net_weights(model)

# Tune your hyperparameters
# 调节超参数

**What's wrong?**. Looking at the visualizations above, we see that the loss is decreasing more or less linearly, which seems to suggest that the learning rate may be too low. Moreover, there is no gap between the training and validation accuracy, suggesting that the model we used has low capacity, and that we should increase its size. On the other hand, with a very large model we would expect to see more overfitting, which would manifest itself as a very large gap between the training and validation accuracy.
**哪里出了问题？** 看上面的可视化，损失大致在呈线性下降，这似乎说明学习率可能太低了。而且，训练准确率和验证准确率之间没有差距，说明我们用的模型容量偏低，应该把它的规模调大。另一方面，模型非常大的话，我们会看到更严重的过拟合，表现为训练准确率和验证准确率之间出现很大的差距。

**Tuning**. Tuning the hyperparameters and developing intuition for how they affect the final performance is a large part of using Neural Networks, so we want you to get a lot of practice. Below, you should experiment with different values of the various hyperparameters, including hidden layer size, learning rate, numer of training epochs, and regularization strength. You might also consider tuning the learning rate decay, but you should be able to get good performance using the default value.
**调参**。调节超参数、并培养出对它们如何影响最终性能的直觉，是使用神经网络的重要一环，所以我们希望你多练一练。下面你应该尝试各种超参数的不同取值，包括隐藏层大小、学习率、训练 epoch 数和正则化强度。你也可以考虑调节学习率衰减，但用默认值应该就能得到不错的表现。

**Approximate results**. You should be aim to achieve a classification accuracy of greater than 48% on the validation set. Our best network gets over 52% on the validation set.
**大致结果**。你的目标是让验证集上的分类准确率超过 48%。我们最好的网络在验证集上能超过 52%。

**Experiment**: You goal in this exercise is to get as good of a result on CIFAR-10 as you can (52% could serve as a reference), with a fully-connected Neural Network. Feel free implement your own techniques (e.g. PCA to reduce dimensionality, or adding dropout, or adding features to the solver, etc.).
**实验**：本练习的目标是用全连接神经网络在 CIFAR-10 上取得尽可能好的结果（52% 可以作为参考）。请随意实现你自己的技巧（例如用 PCA 降维、加入 dropout，或者给 solver 添加特征等）。

In [ ]:
best_model = None



#################################################################################
# TODO: Tune hyperparameters using the validation set. Store your best trained  #
# model in best_model.                                                          #
#                                                                               #
# To help debug your network, it may help to use visualizations similar to the  #
# ones we used above; these visualizations will have significant qualitative    #
# differences from the ones we saw above for the poorly tuned network.          #
#                                                                               #
# Tweaking hyperparameters by hand can be fun, but you might find it useful to  #
# write code to sweep through possible combinations of hyperparameters          #
# automatically like we did on thexs previous exercises.                          #
# TODO:                                                                         #
# 用验证集调节超参数，把训练得到的最佳模型存到 best_model 中。                  #
#                                                                               #
# 为了帮助调试网络，使用与上面类似的可视化可能会有帮助；这些可视化在观感上      #
# 会与上面那个调参很差的网络有明显差别。                                        #
#                                                                               #
# 手工微调超参数可能挺有趣，但你也许会发现，写代码来自动遍历各种超参数组合      #
# 会很有用，就像我们在之前的练习中做的那样。                                    #
#################################################################################

################################################################################
#                              END OF YOUR CODE                                #
################################################################################

# Test your model!
# 测试你的模型！
Run your best model on the validation and test sets. You should achieve above 48% accuracy on the validation set and the test set.
在验证集和测试集上运行你最好的模型。你应当在验证集和测试集上都取得 48% 以上的准确率。

In [ ]:
y_val_pred = np.argmax(best_model.loss(data['X_val']), axis=1)
print('Validation set accuracy: ', (y_val_pred == data['y_val']).mean())

In [ ]:
y_test_pred = np.argmax(best_model.loss(data['X_test']), axis=1)
print('Test set accuracy: ', (y_test_pred == data['y_test']).mean())

In [ ]:
# Save best model
# 保存最佳模型
best_model.save("best_two_layer_net.npy")

## Inline Question 2:
## 课堂提问 2：

Now that you have trained a Neural Network classifier, you may find that your testing accuracy is much lower than the training accuracy. In what ways can we decrease this gap? Select all that apply.
现在你已经训练好了一个神经网络分类器，可能会发现测试准确率比训练准确率低很多。我们可以用哪些办法缩小这个差距？选出所有符合的选项。

1. Train on a larger dataset.
2. Add more hidden units.
3. Increase the regularization strength.
4. None of the above.

1. 在更大的数据集上训练。
2. 增加更多的隐藏单元。
3. 增大正则化强度。
4. 以上都不是。

$\color{blue}{\textit Your Answer:}$
**你的答案：**

$\color{blue}{\textit Your Explanation:}$
**你的解释：**

